# TAEHV 1.5 Streaming Encoder/Decoder Demo

This notebook demonstrates streaming encoding and decoding with TAEHV1.5, which enables:
- **Chunked encoding**: Process long videos without loading everything into memory
- **Autoregressive decoding**: Decode latents one frame at a time for world models
- **O(1) memory**: Stream videos of arbitrary length

The streaming classes maintain temporal state (MemBlocks, TPool) across chunks to ensure consistency.

## Video Downloading

In [ ]:
!python3 -m pip install --quiet yt-dlp

In [ ]:
!yt-dlp --no-warnings 'https://www.youtube.com/shorts/LLXK7ub_QWQ' -o big_cake

In [ ]:
!ffmpeg -y -i "big_cake."* -t 2 -v quiet -hide_banner -stats \
  -vf "crop=min(iw\,ih):min(iw\,ih),scale=256:256" \
  -c:v libx264 -b:v 5M -pix_fmt yuv420p -an -r 30 \
  small_cake.mp4

In [ ]:
from IPython.display import Video, Markdown
import torch as th

Video("small_cake.mp4", html_attributes="playsinline autoplay loop muted", embed=True)

# Helper Functions

In [ ]:
import cv2

class VideoTensorReader:
    def __init__(self, video_file_path):
        self.cap = cv2.VideoCapture(video_file_path)
        assert self.cap.isOpened(), f"Could not load {video_file_path}"
        self.fps = self.cap.get(cv2.CAP_PROP_FPS)
    def __iter__(self):
        return self
    def __next__(self):
        ret, frame = self.cap.read()
        if not ret:
            self.cap.release()
            raise StopIteration
        return th.from_numpy(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)).permute(2, 0, 1)

class VideoTensorWriter:
    def __init__(self, video_file_path, width_height, fps=30):
        self.writer = cv2.VideoWriter(video_file_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, width_height)
        assert self.writer.isOpened(), f"Could not create writer for {video_file_path}"
    def write(self, frame_tensor):
        assert frame_tensor.ndim == 3 and frame_tensor.shape[0] == 3
        self.writer.write(cv2.cvtColor(frame_tensor.permute(1, 2, 0).numpy(), cv2.COLOR_RGB2BGR))
    def __del__(self):
        if hasattr(self, 'writer'): self.writer.release()

# Load TAEHV1.5

In [ ]:
from taehv import TAEHV, StreamingTAEHVEncoder, StreamingTAEHVDecoder

model = TAEHV("taehv1_5.pth").to("cuda", th.float16).eval().requires_grad_(False)
print(f"Model loaded!")
print(f"  Temporal downscale: {model.t_downscale}x (4 frames → 1 latent)")
print(f"  Temporal upscale: {model.t_upscale}x (1 latent → 4 frames)")

# Test 1: Batch Encoding/Decoding (Baseline)

In [ ]:
# Load video
video = th.stack(list(VideoTensorReader("small_cake.mp4")), 0)[None].to("cuda", th.float16).div_(255.0)
video = video[:, :4*((video.shape[1]-1)//4)+1]  # Trim to valid frame count

print(f"Video shape: {video.shape}")

# Batch encode/decode
with th.no_grad():
    latent = model.encode_video(video, parallel=True)
    reconstructed_batch = model.decode_video(latent, parallel=True)

print(f"Latent shape: {latent.shape}")
print(f"Reconstructed shape: {reconstructed_batch.shape}")

# Save
writer = VideoTensorWriter("batch_reconstructed.mp4", reconstructed_batch.shape[-2:][::-1])
for frame in reconstructed_batch.mul(255.0).round_().byte().cpu().squeeze(0):
    writer.write(frame)
del writer

!ffmpeg -y -i batch_reconstructed.mp4 -v quiet -hide_banner -stats -c:v libx264 -b:v 5M -pix_fmt yuv420p -an batch_reconstructed.compressed.mp4
display(Markdown("### Batch Reconstruction"))
display(Video("batch_reconstructed.compressed.mp4", html_attributes="playsinline autoplay loop muted controls", embed=True))

# Test 2: Streaming Encoder (Chunked Encoding)

Process video in small chunks to avoid loading everything into memory.

In [ ]:
streaming_encoder = StreamingTAEHVEncoder(model)
streaming_encoder.reset()

chunk_size = 16  # Process 16 frames at a time
all_latents = []

print(f"Encoding in chunks of {chunk_size} frames...")

with th.no_grad():
    for i in range(0, video.shape[1], chunk_size):
        chunk = video[:, i:i+chunk_size]
        chunk_latent = streaming_encoder.feed_frames(chunk)
        
        if chunk_latent is not None:
            all_latents.append(chunk_latent)
            print(f"  Chunk {i//chunk_size + 1}: {chunk.shape[1]} frames → {chunk_latent.shape[1]} latents")
    
    # Finalize (flush buffers)
    final_latent = streaming_encoder.finalize()
    if final_latent is not None:
        all_latents.append(final_latent)

latent_streaming = th.cat(all_latents, dim=1)
print(f"\nStreaming latent shape: {latent_streaming.shape}")
print(f"Batch latent shape: {latent.shape}")

# Compare latents
latent_diff = (latent - latent_streaming).abs()
print(f"\nLatent difference (batch vs streaming):")
print(f"  Max: {latent_diff.max().item():.6f}")
print(f"  Mean: {latent_diff.mean().item():.6f}")

if latent_diff.max() < 1e-5:
    print("✓ Streaming encoder matches batch encoder perfectly!")

# Test 3: Streaming Decoder (One Latent at a Time)

Decode latents one frame at a time, useful for autoregressive world model generation.

In [ ]:
streaming_decoder = StreamingTAEHVDecoder(model)
streaming_decoder.reset()

all_frames = []

print(f"Decoding {latent_streaming.shape[1]} latents one at a time...")

with th.no_grad():
    for i in range(latent_streaming.shape[1]):
        latent_single = latent_streaming[:, i:i+1]
        frames = streaming_decoder.decode_single_latent(latent_single)
        
        if frames is not None and frames.shape[1] > 0:
            all_frames.append(frames)
            if i % 5 == 0:
                total_frames = sum(f.shape[1] for f in all_frames)
                print(f"  Latent {i:2d} → {frames.shape[1]} frames (total: {total_frames})")

reconstructed_streaming = th.cat(all_frames, dim=1)
print(f"\nStreaming reconstructed shape: {reconstructed_streaming.shape}")
print(f"Batch reconstructed shape: {reconstructed_batch.shape}")

# Save
writer = VideoTensorWriter("streaming_reconstructed.mp4", reconstructed_streaming.shape[-2:][::-1])
for frame in reconstructed_streaming.mul(255.0).round_().byte().cpu().squeeze(0):
    writer.write(frame)
del writer

!ffmpeg -y -i streaming_reconstructed.mp4 -v quiet -hide_banner -stats -c:v libx264 -b:v 5M -pix_fmt yuv420p -an streaming_reconstructed.compressed.mp4
display(Markdown("### Streaming Reconstruction"))
display(Video("streaming_reconstructed.compressed.mp4", html_attributes="playsinline autoplay loop muted controls", embed=True))

# Comparison: Batch vs Streaming

Compare the reconstructions to verify streaming produces identical results.

In [ ]:
min_frames = min(reconstructed_batch.shape[1], reconstructed_streaming.shape[1])

frame_diff = (reconstructed_batch[:, :min_frames] - reconstructed_streaming[:, :min_frames]).abs()

print(f"Frame difference (batch vs streaming):")
print(f"  Max: {frame_diff.max().item():.6f}")
print(f"  Mean: {frame_diff.mean().item():.6f}")

if frame_diff.max() < 1e-3:
    print("\n✓ Streaming decoder matches batch decoder!")
else:
    print("\n✗ Small differences may exist due to frame trimming behavior")

# Side-by-Side Comparison

In [ ]:
display(Markdown("### Original"))
display(Video("small_cake.mp4", html_attributes="playsinline autoplay loop muted controls", embed=True))

display(Markdown("### Batch Reconstruction"))
display(Video("batch_reconstructed.compressed.mp4", html_attributes="playsinline autoplay loop muted controls", embed=True))

display(Markdown("### Streaming Reconstruction"))
display(Video("streaming_reconstructed.compressed.mp4", html_attributes="playsinline autoplay loop muted controls", embed=True))

# Memory Comparison

The key advantage of streaming is memory efficiency. For long videos:

- **Batch mode**: Loads entire video into memory (VRAM requirement scales with video length)
- **Streaming mode**: Processes chunks sequentially (O(1) memory w.r.t. video length)

For a 10-minute 60fps video at 720p:
- Batch mode: ~400GB of memory required
- Streaming mode: Only needs memory for current chunk (~2GB for 128-frame chunks)

# Use Cases

## Streaming Encoder
- Processing long videos from disk without OOM
- Encoding video datasets for training
- Real-time encoding pipelines

## Streaming Decoder
- **Autoregressive world models**: Generate latents one at a time, decode to frames immediately
- **Real-time rendering**: Display frames as they're decoded
- **Interactive generation**: Stream video output as model generates